# Huấn luyện mô hình quét cà vẹt xe (Vehicle Registration Card OCR)

**HƯỚNG DẪN:**
1. Tải file này lên **[Google Colab](https://colab.research.google.com/)**.
2. Bật GPU (Runtime -> Change runtime type -> T4 GPU).
3. Chạy từng ô (cell) từ trên xuống dưới.

In [ ]:
# 1. Cài đặt thư viện
!pip install ultralytics roboflow

### 2. Tải Dataset Cà Vẹt Xe từ Roboflow
Bạn cần tự tạo một dự án trên Roboflow, tải lên các ảnh cà vẹt thực tế của bạn và vẽ khung (bounding box) cho các trường thông tin quan trọng. Lưu ý các classes bạn tạo phải tên là (hoặc tương tự):
- `plate` (hoặc `bien_so`)
- `name` (hoặc `ten_chu_xe`)
- `brand` (hoặc `nhan_hieu`)
- `model` (hoặc `so_loai`)
- `color` (hoặc `mau_son`)

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
import os
from google.colab import files

# Thay đoạn code dưới đây bằng code bạn lấy từ Roboflow Export (Tab YOLOv8)
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT_NAME")
dataset = project.version(1).download("yolov8")

print("Tải dataset hoàn tất!")

### 3. Huấn luyện mô hình YOLOv8n (Bản nhẹ cho Mac)
Vì bạn đang chạy inference trên máy Mac không có card rời, chúng ta sẽ ưu tiên dùng YOLOv8 bản Nano (chạy rất nhanh trên CPU).

In [ ]:
# Tải mô hình YOLOv8 nano mới tinh
model = YOLO('yolov8n.pt')

# Bắt đầu train (Nên train khoảng 100 vòng vì layout cà vẹt thường cố định, dễ học)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=100, imgsz=640, plots=True)

print("Train cà vẹt xong!")

### 4. Tải file `registration.pt` về máy
Copy file này đè vào thư mục `ai_service` với tên là `registration.pt`.

In [ ]:
best_model_path = '/content/runs/detect/train/weights/best.pt'

if os.path.exists(best_model_path):
    # Đổi tên file cho đỡ nhầm lẫn với best.pt của biển số
    os.rename(best_model_path, 'registration.pt')
    print("Đang tải file registration.pt về máy...")
    files.download('registration.pt')
    print("Hãy copy file registration.pt này vào thư mục 'ai_service'.")
else:
    print("Lỗi: Không tìm thấy file model. Train bị lỗi?")